# IND320 – Project Work Part 4: Machine Learning

Name: Alina Shah  
GitHub: https://github.com/59762/IND320-59762  
Streamlit: https://ind320-59762.streamlit.app/

This notebook covers the Jupyter Notebook tasks of part 4:

1. A function that downloads EUR and USD exchange rates from Norges Bank (NB).
2. An investigation of time-zone / timestamp differences between Magasinstatistikk (NVE), ENTSO-E, Open-Meteo and NB.
3. A forecasting function for reservoir filling in one region, built with the `river` framework (SNARIMAX with optional exogenous variables).
4. Tests of the function with different settings, and the choice of default settings for the Streamlit app.

Run the notebook from the root of the repository, from top to bottom. The first run downloads data (Open-Meteo is rate limited, so it can take a few minutes); everything is cached in `data_cache/`, so later runs are fast.

In [3]:
%pip install river

  Using cached river-0.27.0-cp311-cp311-win_amd64.whl.metadata (9.7 kB)
Using cached river-0.27.0-cp311-cp311-win_amd64.whl (2.7 MB)
Note: you may need to restart the kernel to use updated packages.


## Cell 1 – Imports and configuration

In [1]:
# Standard library
import copy, io, json, time, tomllib, warnings
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd
import requests

# Plotting (static plots here so the notebook exports cleanly to PDF; the Streamlit app uses Plotly)
import matplotlib.pyplot as plt

# River: online machine learning (SNARIMAX time-series model + online regressors)
from river import linear_model, optim, preprocessing, time_series, tree

try:
    from IPython.display import display
except ImportError:                      # allows running the same code as a plain script
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# API endpoints
NVE_BASE          = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk"
OPENMETEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
NB_API            = "https://data.norges-bank.no/api/data/EXR"      # Norges Bank exchange-rate dataflow

# Local cache so repeated runs do not hit the APIs again (add data_cache/ to .gitignore)
DATA_CACHE = Path("data_cache")
DATA_CACHE.mkdir(exist_ok=True)

HTTP_HEADERS = {"User-Agent": "IND320-student-project (NMBU)"}
TODAY = pd.Timestamp.today().normalize()
OSLO = "Europe/Oslo"


def cache_is_fresh(path: Path, max_age_days: float) -> bool:
    # True if the cache file exists and is younger than max_age_days
    return path.exists() and (time.time() - path.stat().st_mtime) < max_age_days * 86400


def http_get(url, params=None, retries=4, backoff=2.0, timeout=60, rate_limit_wait=65):
    # GET with retries. Network errors and 5xx errors are retried with exponential back-off,
    # HTTP 429 (rate limit, used by Open-Meteo) waits `rate_limit_wait` seconds,
    # other 4xx errors are raised immediately because retrying will not fix them.
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers=HTTP_HEADERS)
            if r.status_code == 429:
                try:
                    wait = max(int(r.headers.get("Retry-After", 0)), rate_limit_wait)
                except (TypeError, ValueError):
                    wait = rate_limit_wait
                print(f"  HTTP 429 (rate limit) - waiting {wait}s (attempt {attempt}/{retries})")
                last_err = "HTTP 429 rate limit"
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            last_err = e
            status = getattr(getattr(e, "response", None), "status_code", None)
            if status is not None and 400 <= status < 500:
                raise                                   # client error: do not retry
            time.sleep(backoff ** attempt)
    raise ConnectionError(f"Request to {url} failed after {retries} attempts: {last_err}")

print("Setup OK. Cache directory:", DATA_CACHE.resolve())

Setup OK. Cache directory: C:\Users\Alina\OneDrive - Norwegian University of Life Sciences\NMBU\IND320 - Høst 2026\IND320-59762\notebooks\data_cache


## Cell 2 – Currency exchange rates from Norges Bank (NB)

`fetch_nb_exchange_rates()` downloads the daily NB exchange rates (`EXR` dataflow, spot rates `SP`, business-day frequency `B`) for one or more currencies against NOK. NB quotes **NOK per 1 unit of foreign currency**.

Error handling: invalid currency codes, network failures (retried), unexpected response layouts, empty responses and implausible values all give a readable error or warning instead of a cryptic traceback.

In [2]:
class NBDataError(RuntimeError):
    # Raised when the NB response is empty or has an unexpected layout
    pass


def _parse_nb_csv(text: str, currency: str) -> pd.Series:
    # Turn an SDMX-CSV response from NB into a Series (index = date, values = NOK per 1 unit)
    df = pd.read_csv(io.StringIO(text))
    df.columns = [str(c).strip().upper() for c in df.columns]

    missing = {"TIME_PERIOD", "OBS_VALUE"} - set(df.columns)
    if missing:
        raise NBDataError(f"Unexpected NB response layout: {sorted(missing)} missing. "
                          f"Columns found: {list(df.columns)}")
    if df.empty:
        raise NBDataError(f"NB returned no rows for {currency}")

    # Decimal commas are handled in case a locale slips through
    values = pd.to_numeric(df["OBS_VALUE"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    out = pd.Series(values.to_numpy(),
                    index=pd.to_datetime(df["TIME_PERIOD"], errors="coerce"),
                    name=f"{currency.lower()}_nok")
    out = out[~out.index.isna()].sort_index()
    out = out[~out.index.duplicated(keep="last")]
    out.index.name = "date"
    return out


def fetch_nb_exchange_rates(currencies=("EUR", "USD"), start="2000-01-01", end=None,
                            calendar_fill=False, use_cache=True, max_age_days=1.0) -> pd.DataFrame:
    # Download NB exchange rates (NOK per 1 unit of currency) as one wide DataFrame.
    #   currencies    : ISO codes, e.g. ("EUR", "USD")
    #   start / end   : 'YYYY-MM-DD' (end defaults to today)
    #   calendar_fill : False -> only NB business days (default, what NB publishes)
    #                   True  -> every calendar day, weekends/holidays forward-filled
    #   use_cache     : reuse a local CSV younger than max_age_days
    # Returns columns like 'eur_nok', 'usd_nok' with a DatetimeIndex named 'date'.
    currencies = tuple(c.strip().upper() for c in currencies)
    bad = [c for c in currencies if not (len(c) == 3 and c.isalpha())]
    if bad:
        raise ValueError(f"Invalid currency code(s) {bad}; use 3-letter ISO codes such as 'EUR'")
    end = end or TODAY.strftime("%Y-%m-%d")
    if pd.Timestamp(start) > pd.Timestamp(end):
        raise ValueError(f"start ({start}) is after end ({end})")

    cache_file = DATA_CACHE / f"nb_fx_{'-'.join(currencies)}_{start}_{end}.csv"
    if use_cache and cache_is_fresh(cache_file, max_age_days):
        wide = pd.read_csv(cache_file, index_col="date", parse_dates=["date"])
    else:
        series = []
        for cur in currencies:
            url = f"{NB_API}/B.{cur}.NOK.SP"           # B = business days, SP = spot rate
            last_err = None
            for fmt in ("sdmx-csv", "csv"):             # second format only used if the first is rejected
                try:
                    r = http_get(url, params={"format": fmt, "startPeriod": start,
                                              "endPeriod": end, "locale": "en"})
                    series.append(_parse_nb_csv(r.text, cur))
                    break
                except (NBDataError, requests.exceptions.HTTPError) as e:
                    last_err = e
            else:
                raise NBDataError(f"Could not download {cur} from NB: {last_err}")
        wide = pd.concat(series, axis=1).sort_index()
        wide.to_csv(cache_file)

    n_missing = int(wide.isna().sum().sum())
    if n_missing:
        print(f"Warning: {n_missing} missing NB values (kept as NaN, not invented)")
    # Plausibility check: NOK per EUR/USD has stayed roughly between 5 and 15 in the last 25 years
    for col in wide.columns:
        lo, hi = wide[col].min(), wide[col].max()
        if lo < 2 or hi > 30:
            print(f"Warning: {col} outside the plausible range 2-30 (min {lo:.2f}, max {hi:.2f}) - check the data")

    if calendar_fill:
        wide = wide.resample("D").ffill()
    return wide

**Check 1 – parser test (format test, not real data).** The response parser is tested on a tiny hand-made SDMX-CSV sample with obviously dummy values, so the error handling can be verified without a network connection.

In [3]:
SAMPLE_OK = (
    "STRUCTURE,STRUCTURE_ID,ACTION,FREQ,BASE_CUR,QUOTE_CUR,TENOR,DECIMALS,CALCULATED,UNIT_MULT,COLLECTION,TIME_PERIOD,OBS_VALUE\n"
    "DATAFLOW,NB:EXR(1.0),I,B,EUR,NOK,SP,4,false,0,C,2024-01-03,10.0\n"
    "DATAFLOW,NB:EXR(1.0),I,B,EUR,NOK,SP,4,false,0,C,2024-01-02,10.5\n"      # deliberately unsorted
    "DATAFLOW,NB:EXR(1.0),I,B,EUR,NOK,SP,4,false,0,C,2024-01-04,11.0\n"
)
parsed = _parse_nb_csv(SAMPLE_OK, "EUR")
assert parsed.name == "eur_nok" and len(parsed) == 3
assert parsed.index.is_monotonic_increasing and parsed.iloc[0] == 10.5      # sorted by date

for label, bad_text in [("missing column", "FOO,BAR\n1,2\n"),
                        ("no rows", "TIME_PERIOD,OBS_VALUE\n")]:
    try:
        _parse_nb_csv(bad_text, "EUR")
        raise AssertionError(f"{label}: should have raised NBDataError")
    except NBDataError as e:
        print(f"OK - '{label}' gives a readable error: {e}")

try:
    fetch_nb_exchange_rates(("EURO",))
except ValueError as e:
    print("OK - invalid code rejected:", e)
print("Parser and input validation behave as expected.")

OK - 'missing column' gives a readable error: Unexpected NB response layout: ['OBS_VALUE', 'TIME_PERIOD'] missing. Columns found: ['FOO', 'BAR']
OK - 'no rows' gives a readable error: NB returned no rows for EUR
OK - invalid code rejected: Invalid currency code(s) ['EURO']; use 3-letter ISO codes such as 'EUR'
Parser and input validation behave as expected.


**Check 2 – live download.** EUR and USD from 2000 until today, with sanity checks (sorted, no duplicates, only business days, plausible gaps) and a plot.

In [6]:
fx = fetch_nb_exchange_rates(("EUR", "USD"), start="2000-01-01")

display(fx.tail())
display(fx.describe().T)

# Sanity checks on the downloaded data
assert fx.index.is_monotonic_increasing and not fx.index.duplicated().any()
assert (fx > 0).all().all()
weekend_share = (fx.index.dayofweek >= 5).mean()
gaps = fx.index.to_series().diff().dt.days.dropna()
print(f"Rows: {len(fx)} | {fx.index.min().date()} to {fx.index.max().date()}")
print(f"Share of weekend dates: {weekend_share:.1%} | longest gap between observations: {int(gaps.max())} days")

fig, ax = plt.subplots(figsize=(12, 4))
recent = fx.loc[fx.index >= fx.index.max() - pd.DateOffset(years=5)]
ax.plot(recent.index, recent["eur_nok"], label="EUR/NOK", linewidth=1.0)
ax.plot(recent.index, recent["usd_nok"], label="USD/NOK", linewidth=1.0)
ax.set_title("Norges Bank exchange rates, last 5 years (NOK per 1 unit)")
ax.set_ylabel("NOK"); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

NBDataError: Could not download EUR from NB: Unexpected NB response layout: ['OBS_VALUE', 'TIME_PERIOD'] missing. Columns found: ['FREQ;FREQUENCY;BASE_CUR;BASE CURRENCY;QUOTE_CUR;QUOTE CURRENCY;TENOR;TENOR;DECIMALS;CALCULATED;UNIT_MULT;UNIT MULTIPLIER;COLLECTION;COLLECTION INDICATOR;TIME_PERIOD;OBS_VALUE']

## Cell 3 – Data loaders: Magasinstatistikk (NVE) and Open-Meteo

The reservoir data and the weather data are downloaded the same way as in part 3, wrapped in cached functions. The weather is downloaded as **daily** ERA5 values (instead of hourly) because the models use weekly/monthly resolution; this keeps the number of API calls small.

In [ ]:
NVE_RENAME = {
    "dato_Id": "date", "omrType": "area_type", "omrnr": "area_nr",
    "iso_aar": "year", "iso_uke": "week", "fyllingsgrad": "fill_percentage",
    "kapasitet_TWh": "capacity_TWh", "fylling_TWh": "fill_TWh",
    "neste_Publiseringsdato": "next_publish_date",
    "fyllingsgrad_forrige_uke": "prev_week_fill_pct", "endring_fyllingsgrad": "change_fill_pct",
}


def load_nve_raw(use_cache=True, max_age_days=1.0) -> pd.DataFrame:
    # All public Magasinstatistikk data (all area types) with English column names
    path = DATA_CACHE / "nve_magasinstatistikk.csv"
    if use_cache and cache_is_fresh(path, max_age_days):
        return pd.read_csv(path, parse_dates=["date"])
    r = http_get(f"{NVE_BASE}/HentOffentligData", timeout=120)
    df = pd.DataFrame(r.json()).rename(columns=NVE_RENAME)
    if df.empty or "date" not in df.columns:
        raise ValueError("NVE returned an empty or unexpected response")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = (df.dropna(subset=["date", "area_type", "area_nr"])
            .sort_values(["area_type", "area_nr", "date"]).reset_index(drop=True))
    df.to_csv(path, index=False)
    return df


def get_fill_series(nve: pd.DataFrame, area_type="EL", area_nr=3) -> pd.Series:
    # Weekly fill percentage (0-100) for ONE region, as a Series named 'y' with a date index
    d = nve[(nve["area_type"] == area_type) & (nve["area_nr"].astype(float) == float(area_nr))]
    if d.empty:
        avail = {t: sorted(int(n) for n in g["area_nr"].unique()) for t, g in nve.groupby("area_type")}
        raise ValueError(f"No Magasinstatistikk data for {area_type} {area_nr}. Available areas: {avail}")
    s = pd.to_numeric(d.set_index("date")["fill_percentage"], errors="coerce").sort_index()
    s = s[~s.index.duplicated(keep="last")].dropna()
    if s.max() <= 1.5:                    # the API may give a fraction (0-1); the models use percent
        s = s * 100.0
    s.name = "y"
    return s


nve_raw = load_nve_raw()
fill_weekly = get_fill_series(nve_raw, "EL", 3)
print(f"NVE rows: {len(nve_raw)} | EL 3 weekly series: {len(fill_weekly)} obs, "
      f"{fill_weekly.index.min().date()} to {fill_weekly.index.max().date()}")
fill_weekly.tail()

In [ ]:
# Approximate bounding boxes (lon_min, lon_max, lat_min, lat_max) per electricity area, as in part 3
AREA_BBOX = {
    1: (10.4, 12.1, 59.0, 60.5),   # NO1 Eastern Norway
    2: ( 5.0,  9.0, 57.9, 60.0),   # NO2 South-western Norway
    3: ( 9.0, 17.0, 62.0, 65.5),   # NO3 Central Norway
    4: (13.5, 28.5, 65.5, 71.2),   # NO4 Northern Norway
    5: ( 4.9,  8.8, 60.0, 62.5),   # NO5 Western Norway
}

# Daily Open-Meteo variables and how they are aggregated to weeks/months
WEATHER_VARS = {
    "temperature_2m_mean": "mean",   # degC
    "precipitation_sum":   "sum",    # mm
    "snowfall_sum":        "sum",    # cm
    "wind_speed_10m_max":  "mean",   # km/h
}


def area_points(area_type="EL", area_nr=3, n_points=6, seed=42):
    # Sample points inside the area's bounding box (reproducible with the seed).
    # 'NO' (whole country) spreads the points over all five electricity areas.
    rng = np.random.default_rng(seed)
    if area_type == "EL":
        if int(area_nr) not in AREA_BBOX:
            raise ValueError(f"No bounding box for EL {area_nr}; choose from {list(AREA_BBOX)}")
        boxes, per_box = [AREA_BBOX[int(area_nr)]], n_points
    elif area_type == "NO":
        boxes, per_box = list(AREA_BBOX.values()), int(np.ceil(n_points / len(AREA_BBOX)))
    else:
        raise ValueError("Weather data is only available for area types 'EL' and 'NO' (VASS has no geometry here)")
    pts = []
    for lon0, lon1, lat0, lat1 in boxes:
        pts += list(zip(rng.uniform(lon0, lon1, per_box).round(3), rng.uniform(lat0, lat1, per_box).round(3)))
    return [(float(lon), float(lat)) for lon, lat in pts]


def fetch_weather_daily(area_type="EL", area_nr=3, start="2005-01-01", end=None, n_points=6,
                        variables=tuple(WEATHER_VARS), tz=OSLO, seed=42, use_cache=True, max_age_days=7.0):
    # Area-average daily ERA5 weather from Open-Meteo (index = local calendar date in `tz`).
    # Each location is cached separately, so an interrupted download continues where it stopped.
    end = end or (TODAY - pd.Timedelta(days=6)).strftime("%Y-%m-%d")     # ERA5 lags a few days
    variables = tuple(variables)
    unknown = [v for v in variables if v not in WEATHER_VARS]
    if unknown:
        raise ValueError(f"Unknown weather variable(s) {unknown}; choose from {list(WEATHER_VARS)}")

    frames, failed = [], []
    points = area_points(area_type, area_nr, n_points, seed)
    for i, (lon, lat) in enumerate(points, 1):
        f = DATA_CACHE / f"om_daily_{lat:.3f}_{lon:.3f}_{start}_{end}_{'-'.join(variables)}.csv"
        if use_cache and cache_is_fresh(f, max_age_days):
            frames.append(pd.read_csv(f, index_col="date", parse_dates=["date"]))
            continue
        params = {"latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
                  "daily": ",".join(variables), "timezone": tz, "models": "era5", "timeformat": "iso8601"}
        try:
            j = http_get(OPENMETEO_ARCHIVE, params=params, timeout=120).json()
            daily = j.get("daily") or {}
            if "time" not in daily:
                raise ValueError(f"no 'daily' block in response: {str(j)[:150]}")
            d = pd.DataFrame(daily).rename(columns={"time": "date"})
            d["date"] = pd.to_datetime(d["date"])
            d = d.set_index("date")[list(variables)]
            d.to_csv(f)
            frames.append(d)
            print(f"  point {i}/{len(points)} downloaded ({lat:.2f}N, {lon:.2f}E)")
            time.sleep(1.0)                                                   # be polite to the free API
        except Exception as e:                                                # keep going, report at the end
            failed.append((lat, lon, str(e)[:120]))
            print(f"  point {i}/{len(points)} FAILED: {str(e)[:120]}")

    if not frames:
        raise ConnectionError(f"Open-Meteo download failed for all {len(points)} points. Last error: {failed[-1][2]}")
    if failed:
        print(f"Warning: {len(failed)} of {len(points)} points failed; the area average uses the rest")
    area_mean = pd.concat(frames).groupby(level=0).mean().sort_index()
    area_mean.index.name = "date"
    return area_mean


weather_daily = fetch_weather_daily("EL", 3, start="2005-01-01")
print(f"Weather: {len(weather_daily)} days, {weather_daily.index.min().date()} to {weather_daily.index.max().date()}")
print("Missing values per column:", weather_daily.isna().sum().to_dict())
weather_daily.tail()

## Cell 4 – Time zones and timestamps across the four sources

The four sources describe time in different ways, and the differences matter as soon as they are merged into one table:

| Source | What a timestamp means (documented) | What is checked below |
|---|---|---|
| Magasinstatistikk (NVE) | A calendar date without clock time or time zone, one value per week | Which weekday the dates fall on, time-of-day part, time zone, spacing, publication day |
| ENTSO-E | UTC timestamps (XML periods start with `Z`), 15/30/60-minute resolution | How many UTC hours a Norwegian local day has around daylight-saving time |
| Open-Meteo | Wall-clock time in the *requested* `timezone` (default GMT), without offset in the string; daily values are cut at local midnight of that time zone | Number of hourly rows on the DST days and whether timestamps repeat |
| Norges Bank | A calendar date, business days only | Weekend dates, gaps (holidays) |

The cells below check each point against the real data, and the findings are summarised after the checks.

In [ ]:
# 4a) Magasinstatistikk: what kind of timestamp is `date`?
WEEKDAY_CODES = ["MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN"]

idx = fill_weekly.index
wd_counts = pd.Series(idx.dayofweek).map(lambda i: WEEKDAY_CODES[i]).value_counts()
print("Weekday of NVE dates (EL 3):", wd_counts.to_dict())
print("Time zone information on the index:", idx.tz)
print("Any time-of-day component:", bool((idx != idx.normalize()).any()))
print("Days between observations:", idx.to_series().diff().dt.days.value_counts().sort_index().to_dict())

# Same check for ALL areas, to be sure the weekday is a property of the dataset and not of one region
all_wd = pd.Series(nve_raw["date"].dt.dayofweek).map(lambda i: WEEKDAY_CODES[i]).value_counts(normalize=True)
print("Weekday share, all areas:", all_wd.round(3).to_dict())

# The weekday most observations fall on defines the week anchor used for weekly resampling below
NVE_WEEK_ANCHOR = wd_counts.idxmax()
print(f"\n-> weekly anchor used from here on: W-{NVE_WEEK_ANCHOR}")

# Publication day: when is each weekly value actually released? (matters for real-life forecasting)
if "next_publish_date" in nve_raw.columns:
    pub = pd.to_datetime(nve_raw["next_publish_date"], errors="coerce").dropna()
    if len(pub):
        print("Weekday of 'next_publish_date':",
              pd.Series(pub.dt.dayofweek).map(lambda i: WEEKDAY_CODES[i]).value_counts().to_dict())

In [ ]:
# 4b) Open-Meteo: hourly data around the two daylight-saving switches in 2025 (Oslo: 30 March and 26 October)
def fetch_hourly_check(lat, lon, start, end, tz):
    params = {"latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
              "hourly": "temperature_2m", "timezone": tz, "models": "era5", "timeformat": "iso8601"}
    j = http_get(OPENMETEO_ARCHIVE, params=params, timeout=60).json()
    h = j["hourly"]
    meta = {k: j.get(k) for k in ("timezone", "timezone_abbreviation", "utc_offset_seconds")}
    return pd.DataFrame({"time_str": h["time"], "temp": h["temperature_2m"]}), meta


lon0, lat0 = area_points("EL", 3, 1)[0]
DST_CASES = {"autumn: clocks back (2025-10-26)": ("2025-10-25", "2025-10-27"),
             "spring: clocks forward (2025-03-30)": ("2025-03-29", "2025-03-31")}
om_dst_report = {}

for name, (s, e) in DST_CASES.items():
    h_osl, meta_osl = fetch_hourly_check(lat0, lon0, s, e, OSLO)
    h_utc, meta_utc = fetch_hourly_check(lat0, lon0, s, e, "GMT")
    per_day = pd.to_datetime(h_osl["time_str"]).dt.date.value_counts().sort_index()
    n_dupe = int(h_osl["time_str"].duplicated().sum())
    print(f"=== {name}")
    print("  timezone=Europe/Oslo ->", meta_osl)
    print("  rows per local date    :", {str(k): int(v) for k, v in per_day.items()})
    print("  repeated time strings  :", n_dupe)
    print("  timezone=GMT         ->", meta_utc, "| rows:", len(h_utc))

    # Do the two requests describe the same instants? Convert the local strings to UTC and compare.
    try:
        local = pd.to_datetime(h_osl["time_str"])
        utc_from_local = local.dt.tz_localize(OSLO, ambiguous="infer", nonexistent="shift_forward").dt.tz_convert("UTC")
        a = pd.Series(h_osl["temp"].to_numpy(), index=utc_from_local.dt.tz_localize(None))
        b = pd.Series(h_utc["temp"].to_numpy(), index=pd.to_datetime(h_utc["time_str"]))
        common = a.index.intersection(b.index)
        diff = (a.loc[common] - b.loc[common]).abs().max()
        print(f"  local->UTC conversion matches the GMT request on {len(common)} instants, max |diff| = {diff:.3f}")
        om_dst_report[name] = dict(rows_per_day=per_day.to_dict(), repeated=n_dupe, max_abs_diff=float(diff))
    except Exception as err:
        print("  could not map local strings to UTC unambiguously:", str(err)[:120])
        om_dst_report[name] = dict(rows_per_day=per_day.to_dict(), repeated=n_dupe, max_abs_diff=None)

In [ ]:
# 4c) ENTSO-E is in UTC. What does that mean for a Norwegian *local* day?
# Pure pandas demonstration (no data needed): how many UTC hours make up each full Norwegian local day?
for label, (a, z) in {"spring switch": ("2025-03-28", "2025-04-02"), "autumn switch": ("2025-10-24", "2025-10-29")}.items():
    local_hours = pd.date_range(a, z, freq="h", tz=OSLO, inclusive="left")          # whole local days
    utc_hours = pd.Series(1, index=local_hours.tz_convert("UTC"))
    n = utc_hours.groupby(local_hours.date).sum()
    print(f"Hours per Norwegian local day, {label}:", {str(k): int(v) for k, v in n.items()})


def load_secrets(path=".streamlit/secrets.toml"):
    p = Path(path)
    if p.exists():
        with open(p, "rb") as f:
            return tomllib.load(f)
    return {}


def check_entsoe_in_mongo():
    # Look at the processed ENTSO-E data from part 2 (MongoDB). Skipped if MongoDB is not reachable.
    try:
        from pymongo import MongoClient
        import os
        uri = load_secrets().get("mongodb_uri") or os.getenv("MONGODB_URI")
        if not uri:
            print("No MongoDB URI found - skipping the database check")
            return None
        client = MongoClient(uri, serverSelectionTimeoutMS=8000)
        col = client["ind320"]["entsoe_hourly"]
        first = col.find_one(sort=[("ts", 1)], projection={"_id": 0})
        last = col.find_one(sort=[("ts", -1)], projection={"_id": 0})
        print("First stored hour:", first["ts"], "| last stored hour:", last["ts"])
        print("Stored timestamps are timezone-naive and represent UTC (part 2 converts everything to UTC before storing).")
        return first, last
    except Exception as err:
        print("MongoDB check skipped:", str(err)[:150])
        return None

_ = check_entsoe_in_mongo()

In [ ]:
# 4d) Norges Bank: date-only, business days only
nb_wd = pd.Series(fx.index.dayofweek).map(lambda i: WEEKDAY_CODES[i]).value_counts().reindex(WEEKDAY_CODES, fill_value=0)
print("Weekday of NB observations:", nb_wd.to_dict())
print("Date-only timestamps (no clock time):", bool((fx.index == fx.index.normalize()).all()))
cal = pd.date_range(fx.index.min(), fx.index.max(), freq="D")
print(f"Calendar days without an NB rate: {len(cal) - len(fx)} of {len(cal)} "
      f"({(len(cal) - len(fx)) / len(cal):.1%}) -> weekends and Norwegian bank holidays")

**Alignment helpers.** The rules that follow from the checks are implemented once and reused by the dataset builder:

- **Everything hourly is converted to a timezone-aware index first** (ENTSO-E: UTC → `Europe/Oslo`), and only then aggregated to days/weeks/months, so a local day with 23 or 25 hours is averaged correctly.
- **Daily weather is requested with `timezone=Europe/Oslo`**, so its day boundaries are the same as the Norwegian calendar dates used by NVE. If hourly weather is ever needed, it is requested in GMT and converted.
- **NVE weeks** are labelled by their week-ending date (`W-<anchor>` found in 4a); monthly values use the last weekly observation of the month, because filling is a *stock* (a state at a point in time), whereas precipitation and snowfall are *flows* (summed).
- **NB rates** only exist on business days, so they are forward-filled to calendar days before being averaged to weeks/months.

In [ ]:
def hourly_utc_to_daily_oslo(s: pd.Series, how="mean") -> pd.Series:
    # Hourly series (UTC; naive index is interpreted as UTC) -> daily series on Norwegian local dates.
    # resample('D') on a tz-aware index respects daylight-saving time (23/25-hour days).
    s = s.copy()
    idx = pd.DatetimeIndex(s.index)
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    s.index = idx.tz_convert(OSLO)
    out = s.resample("D").agg(how)
    out.index = out.index.tz_localize(None)          # naive local date, like NVE and NB
    return out


def local_wallclock_to_utc(index_local) -> pd.DatetimeIndex:
    # Wall-clock Oslo timestamps (e.g. Open-Meteo with timezone=Europe/Oslo) -> naive UTC.
    # ambiguous='infer' handles the repeated hour in autumn when the hours are in order.
    return (pd.DatetimeIndex(index_local).tz_localize(OSLO, ambiguous="infer", nonexistent="shift_forward")
            .tz_convert("UTC").tz_localize(None))


# Unit tests of the helpers on the real DST days (constant 1 MW every hour)
h = pd.Series(1.0, index=pd.date_range("2025-03-28", "2025-04-02", freq="h", inclusive="left"))
daily_energy = hourly_utc_to_daily_oslo(h, "sum")
assert daily_energy.loc["2025-03-30"] == 23 and daily_energy.loc["2025-03-31"] == 24
h = pd.Series(1.0, index=pd.date_range("2025-10-24", "2025-10-29", freq="h", inclusive="left"))
daily_energy = hourly_utc_to_daily_oslo(h, "sum")
assert daily_energy.loc["2025-10-26"] == 25
assert hourly_utc_to_daily_oslo(h, "mean").eq(1.0).all()          # mean power is unaffected by day length
wall = pd.DatetimeIndex(["2025-10-26 01:00", "2025-10-26 02:00", "2025-10-26 02:00", "2025-10-26 03:00"])
assert list(local_wallclock_to_utc(wall)) == list(pd.DatetimeIndex(
    ["2025-10-25 23:00", "2025-10-26 00:00", "2025-10-26 01:00", "2025-10-26 02:00"]))
print("Alignment helpers pass their DST tests (23-hour day, 25-hour day, repeated autumn hour).")

In [ ]:
# Observed facts collected in one table (the values come from the checks above)
tz_summary = pd.DataFrame([
    {"source": "Magasinstatistikk (NVE)",
     "time reference": f"calendar date, no time zone, weekly (weekday: {NVE_WEEK_ANCHOR})",
     "alignment rule": f"week label = week-ending date (W-{NVE_WEEK_ANCHOR}); month = last weekly value"},
    {"source": "ENTSO-E",
     "time reference": "UTC (stored as naive UTC), hourly after part 2",
     "alignment rule": "tz_localize('UTC') -> tz_convert('Europe/Oslo') -> then resample to day/week/month"},
    {"source": "Open-Meteo",
     "time reference": "wall-clock time in the requested timezone (here Europe/Oslo for daily data)",
     "alignment rule": "daily with timezone=Europe/Oslo (same local dates as NVE); hourly in GMT then convert"},
    {"source": "Norges Bank",
     "time reference": f"calendar date, business days only ({(fx.index.dayofweek >= 5).mean():.0%} weekend dates)",
     "alignment rule": "forward-fill to calendar days, then average to week/month"},
])
display(tz_summary)

**Findings (time zones).** Fill in / confirm against the outputs above after running the notebook:

- **NVE and NB use plain dates**, so they have no time-zone problem of their own. NVE has one value per week on a fixed weekday (printed in 4a) and NB only has business days. NVE therefore defines the weekly grid, and NB has to be forward-filled.
- **ENTSO-E is in UTC**, Norwegian days are not: a local day is 23 h long at the spring switch and 25 h at the autumn switch (4c). Aggregating UTC data by UTC date would shift up to 1–2 hours of every day into the neighbouring day and distort daily sums. The solution is `hourly_utc_to_daily_oslo()`.
- **Open-Meteo depends on the `timezone` parameter**. With `Europe/Oslo` the hourly rows are local wall-clock strings, so the DST days have 23 or 25 rows and (4b) the autumn hour may be repeated in the strings, which makes naive parsing ambiguous. Requesting `GMT` avoids that. For daily values the Oslo time zone is the right choice, because the daily totals then use the same local dates as NVE.
- **Publication lag**: NVE's weekly value is published some days after the week ends (see the publication weekday in 4a), so a real-time forecast is made from a slightly outdated last observation. In the notebook, the forecast origin is the last *complete* period for which every selected source has data.

## Cell 5 – Building the modelling dataset

`build_dataset()` merges the reservoir series with the selected exogenous sources on a common time grid:

- **Resolution**: `"M"` monthly (m = 12) or `"W"` weekly (m = 52, anchored on the NVE week-ending weekday).
- **Target `y`**: fill percentage (state) → last weekly value in each month.
- **Exogenous variables** are catalogued by source: weather (Open-Meteo), currency (NB) and cross-border flow (ENTSO-E). Weather flows are summed (precipitation, snowfall), the others averaged.
- Incomplete last periods are dropped, so no partial month is ever treated as a full one.

In [ ]:
RESOLUTIONS = {
    "M": {"label": "monthly", "m": 12, "per_year": 12, "step": 2},     # step = distance between backtest origins
    "W": {"label": "weekly",  "m": 52, "per_year": 52, "step": 8},
}

# Exogenous catalogue: name -> (source, aggregation to week/month)
EXOG_CATALOG = {
    "temperature_2m_mean":  ("weather",  "mean"),
    "precipitation_sum":    ("weather",  "sum"),
    "snowfall_sum":         ("weather",  "sum"),
    "wind_speed_10m_max":   ("weather",  "mean"),
    "eur_nok":              ("currency", "mean"),
    "usd_nok":              ("currency", "mean"),
    "entsoe_net_import_mw": ("entsoe",   "mean"),
}
# How the future value of an exogenous variable is assumed when forecasting (it is unknown!)
#   climatology: average of the same month/week in the history; persist: last known value
FUTURE_MODE = {"weather": "climatology", "currency": "persist", "entsoe": "persist"}


def period_rule(res):
    return "MS" if res == "M" else f"W-{NVE_WEEK_ANCHOR}"


def period_key(index, res):
    # Month number (monthly) or ISO week number capped at 52 (weekly): used for climatologies
    index = pd.DatetimeIndex(index)
    return index.month.to_numpy() if res == "M" else np.minimum(index.isocalendar().week.to_numpy(), 52)


def future_index(last, horizon, res):
    return pd.date_range(pd.Timestamp(last), periods=horizon + 1, freq=period_rule(res))[1:]


def aggregate_fill(fill_weekly: pd.Series, res: str) -> pd.Series:
    s = fill_weekly.sort_index()
    if res == "M":
        y = s.resample("MS").last()
        # drop the last month if the newest observation is not from its final week
        if (y.index[-1] + pd.offsets.MonthEnd(0)) - s.index.max() > pd.Timedelta(days=6):
            y = y.iloc[:-1]
    else:
        y = s.resample(period_rule("W"), label="right", closed="right").last()
    return y.interpolate(limit=2, limit_area="inside")


def aggregate_daily(daily: pd.DataFrame, res: str) -> pd.DataFrame:
    # Daily (local dates) -> week/month with the aggregation given in EXOG_CATALOG; incomplete last period dropped
    kw = {} if res == "M" else {"label": "right", "closed": "right"}
    r = daily.resample(period_rule(res), **kw)
    cols = {}
    for c in daily.columns:
        how = EXOG_CATALOG[c][1]
        cols[c] = r[c].sum(min_count=1) if how == "sum" else r[c].mean()
    out = pd.concat(cols, axis=1)
    last_day = daily.dropna(how="all").index.max()
    period_end = out.index + pd.offsets.MonthEnd(0) if res == "M" else out.index
    return out[period_end <= last_day]


def build_dataset(fill_weekly, resolution="M", exog=(), weather_daily=None, fx_daily=None, entsoe_daily=None):
    # Returns a DataFrame with column 'y' (+ the selected exogenous columns) on a regular week/month grid.
    if resolution not in RESOLUTIONS:
        raise ValueError(f"resolution must be one of {list(RESOLUTIONS)}")
    exog = tuple(exog)
    unknown = [c for c in exog if c not in EXOG_CATALOG]
    if unknown:
        raise ValueError(f"Unknown exogenous variable(s) {unknown}. Choose from {list(EXOG_CATALOG)}")

    ds = aggregate_fill(fill_weekly, resolution).to_frame("y")
    frames = {"weather": weather_daily, "currency": fx_daily, "entsoe": entsoe_daily}
    for source in ("weather", "currency", "entsoe"):
        cols = [c for c in exog if EXOG_CATALOG[c][0] == source]
        if not cols:
            continue
        daily = frames[source]
        if daily is None:
            raise ValueError(f"Variable(s) {cols} need the '{source}' source, but no data for it was supplied")
        missing = [c for c in cols if c not in daily.columns]
        if missing:
            raise ValueError(f"{missing} not found in the {source} data (columns: {list(daily.columns)})")
        daily = daily[cols]
        if source == "currency":
            daily = daily.resample("D").ffill()       # business days -> calendar days (see Cell 4)
        ds = ds.join(aggregate_daily(daily, resolution), how="left")

    # Keep the period where y and all selected variables exist; fill small interior gaps only
    cols = ["y", *exog]
    first = ds[cols].dropna().index.min()
    last = ds[cols].dropna().index.max()
    if pd.isna(first):
        raise ValueError("The selected sources have no overlapping period with the reservoir data")
    ds = ds.loc[first:last, cols]
    n_gap = int(ds.isna().sum().sum())
    if n_gap:
        print(f"Note: {n_gap} interior missing value(s) interpolated")
        ds = ds.interpolate(limit_area="inside").bfill().ffill()
    ds = ds.astype(float)
    ds.attrs["resolution"] = resolution
    return ds


# Quick look at the two resolutions without exogenous data
for res in ("M", "W"):
    d = build_dataset(fill_weekly, res)
    print(f"{RESOLUTIONS[res]['label']:8s}: {len(d):5d} rows, {d.index.min().date()} to {d.index.max().date()}")

## Cell 6 – Forecasting function with River (SNARIMAX)

River's `time_series.SNARIMAX` is an online (streaming) seasonal ARIMA-type model: every observation is learned with `learn_one(y, x)` and `forecast(horizon, xs)` extrapolates `horizon` steps ahead using the model's own earlier forecasts as lags (**dynamic forecasting**). The model orders are
`p, d, q` (non-seasonal AR, differencing, MA) and `sp, sd, sq` (seasonal), with season length `m` (12 for monthly, 52 for weekly data). The coefficients are fitted by an online regressor, so the *lag-based machine-learning part* is selectable:

| `regressor` | River model | Notes |
|---|---|---|
| `"linear"` | `StandardScaler → LinearRegression` (SGD) | `lr`, `l2`, `intercept_lr` can be tuned; supports quantile loss for intervals |
| `"pa"` | `StandardScaler → PARegressor` | passive-aggressive, parameter `C` |
| `"tree"` | `StandardScaler → HoeffdingAdaptiveTreeRegressor` | non-linear, cannot extrapolate beyond seen values |

**Exogenous variables** (weather, currency, ENTSO-E flow) enter as extra features, optionally with lags (`exog_lags`). They are unknown in the future, so for the forecast they are replaced by a *climatology* (weather: mean of the same month/week in the history) or by the last known value (currency, flow). A diagnostic mode (`oracle`) uses the true future values, to show how much a perfect weather forecast could add.

**Target transformation (`target`)**: the filling has a strong seasonal cycle, and a recursive multi-step forecast of a *level* series is fragile (small coefficient errors are repeated and amplified over the horizon). With `target="anomaly"` the climatology (average of the same month/week over the training window) is subtracted, SNARIMAX models the *anomaly*, and the climatology is added back to the forecast. Weather variables are converted to anomalies in the same way, so "future weather = climatology" simply means "future anomaly = 0". `target="level"` models the raw series (and then needs seasonal terms `sp`/`sd`). Both are tested in Cell 7.

**Confidence intervals** (bonus): with `intervals=(0.1, 0.9)` two extra models are trained with `optim.losses.Quantile` and give a lower and an upper forecast.

In [ ]:
def load_entsoe_net_import_daily(zone="NO3") -> pd.DataFrame:
    # Net import (into Norway minus out of Norway, MW) for one NO zone from the part-2 MongoDB data,
    # aggregated to Norwegian local days (see Cell 4). Only 2024-2025 exist in the database.
    import os
    from pymongo import MongoClient
    uri = load_secrets().get("mongodb_uri") or os.getenv("MONGODB_URI")
    if not uri:
        raise ValueError("No MongoDB URI found in .streamlit/secrets.toml or the environment")
    col = MongoClient(uri, serverSelectionTimeoutMS=10000)["ind320"]["entsoe_hourly"]
    rows = list(col.find({"corridor": {"$regex": f"^{zone} - "}}, {"_id": 0, "ts": 1, "direction": 1, "flow_mw": 1}))
    if not rows:
        raise ValueError(f"No ENTSO-E rows for zone {zone} in MongoDB")
    df = pd.DataFrame(rows)
    df["signed"] = np.where(df["direction"] == "Into Norway", df["flow_mw"], -df["flow_mw"])
    hourly = df.groupby("ts")["signed"].sum().sort_index()
    return hourly_utc_to_daily_oslo(hourly, "mean").to_frame("entsoe_net_import_mw")


def make_regressor(kind="linear", lr=0.01, l2=0.0, intercept_lr=0.1, loss="squared", quantile=0.5, C=1.0, seed=42):
    # Online regressor used inside SNARIMAX. The StandardScaler matters: lags and exogenous
    # variables have very different scales and SGD is sensitive to that.
    if kind == "linear":
        loss_fn = optim.losses.Quantile(alpha=quantile) if loss == "quantile" else optim.losses.Squared()
        reg = linear_model.LinearRegression(optimizer=optim.SGD(lr), loss=loss_fn, l2=l2, intercept_lr=intercept_lr)
    elif kind == "pa":
        reg = linear_model.PARegressor(C=C)
    elif kind == "tree":
        reg = tree.HoeffdingAdaptiveTreeRegressor(seed=seed)
    else:
        raise ValueError("regressor must be 'linear', 'pa' or 'tree'")
    return preprocessing.StandardScaler() | reg


@dataclass
class ForecastResult:
    forecast: pd.Series
    lower: pd.Series | None
    upper: pd.Series | None
    origin: pd.Timestamp          # last observation used for training
    config: dict
    fit_seconds: float
    n_train: int
    resolution: str


def project_future_exog(hist, exog, horizon, res, mode="auto", actual_future=None):
    # Assumed future values of the exogenous variables (they are unknown when forecasting)
    idx = future_index(hist.index[-1], horizon, res)
    fut = pd.DataFrame(index=idx, columns=list(exog), dtype=float)
    for c in exog:
        if mode == "oracle":
            if actual_future is None or len(actual_future) < horizon:
                raise ValueError("oracle mode needs the true future values")
            fut[c] = actual_future[c].to_numpy()[:horizon]
        elif FUTURE_MODE[EXOG_CATALOG[c][0]] == "climatology":
            clim = hist[c].groupby(period_key(hist.index, res)).mean()
            vals = pd.Series(period_key(idx, res)).map(clim).to_numpy(dtype=float)
            fut[c] = np.where(np.isnan(vals), hist[c].mean(), vals)
        else:
            fut[c] = hist[c].iloc[-1]                       # persistence
    return fut


def make_features(hist, fut, exog, lags):
    # Feature dicts for River: current exogenous values + optional lagged copies
    if not exog:
        return None, None
    base = pd.concat([hist[list(exog)], fut[list(exog)]])
    feats = base.copy()
    for lag in lags:
        for c in exog:
            feats[f"{c}_lag{lag}"] = base[c].shift(lag)
    rec = feats.bfill().to_dict("records")                   # first rows have no lag: back-filled
    return rec[:len(hist)], rec[len(hist):]


def _train(model, y, X, epochs):
    for _ in range(epochs):
        for t in range(len(y)):
            model.learn_one(float(y[t]), x=None if X is None else X[t])


def fit_and_forecast(ds, horizon, p=2, d=0, q=0, sp=1, sd=0, sq=0, regressor="linear", lr=0.01, l2=0.0,
                     intercept_lr=0.1, C=1.0, epochs=1, exog=(), exog_lags=(), intervals=None, target="anomaly",
                     exog_future="auto", actual_future=None, resolution=None, seed=42) -> ForecastResult:
    # Train SNARIMAX on ds (last row = forecast origin) and forecast `horizon` periods ahead.
    res = resolution or ds.attrs.get("resolution")
    if res not in RESOLUTIONS:
        raise ValueError("resolution must be 'M' or 'W'")
    if ds["y"].isna().any():
        raise ValueError("The target series contains missing values")
    if horizon < 1:
        raise ValueError("horizon must be at least 1")
    exog = tuple(exog)
    m = RESOLUTIONS[res]["m"]
    y = ds["y"].to_numpy(float)

    if target not in ("anomaly", "level"):
        raise ValueError("target must be 'anomaly' or 'level'")
    idx = future_index(ds.index[-1], horizon, res)
    key_h, key_f = period_key(ds.index, res), period_key(idx, res)

    # Target: subtract the training-window climatology (anomaly mode) so SNARIMAX only models the deviation
    if target == "anomaly":
        clim_y = ds["y"].groupby(key_h).mean()
        base_h = clim_y.reindex(key_h).to_numpy(float)
        base_f = pd.Series(key_f).map(clim_y).fillna(ds["y"].mean()).to_numpy(float)
        y_fit = y - base_h
    else:
        base_f = np.zeros(horizon)
        y_fit = y

    # Exogenous variables (assumed future values; weather as anomalies in anomaly mode)
    Xh = Xf = None
    if exog:
        fut = project_future_exog(ds, exog, horizon, res, exog_future, actual_future)
        hx = ds[list(exog)].copy()
        if target == "anomaly":
            for c_ in exog:
                if FUTURE_MODE[EXOG_CATALOG[c_][0]] == "climatology":
                    clim = ds[c_].groupby(key_h).mean()
                    hx[c_] = ds[c_].to_numpy(float) - clim.reindex(key_h).to_numpy(float)
                    fut[c_] = fut[c_].to_numpy(float) - pd.Series(key_f).map(clim).fillna(ds[c_].mean()).to_numpy(float)
        Xh, Xf = make_features(hx, fut, exog, tuple(exog_lags))

    def new_model(**reg_kw):
        return time_series.SNARIMAX(p=p, d=d, q=q, m=m, sp=sp, sd=sd, sq=sq,
                                    regressor=make_regressor(regressor if not reg_kw else "linear",
                                                             lr=lr, l2=l2, intercept_lr=intercept_lr, C=C, seed=seed, **reg_kw))

    t0 = time.perf_counter()
    model = new_model()
    _train(model, y_fit, Xh, epochs)
    point = np.clip(np.array(model.forecast(horizon, xs=Xf), dtype=float) + base_f, 0, 100)   # fill percentage stays in 0-100
    lower = upper = None
    if intervals:
        if regressor != "linear":
            raise ValueError("intervals use quantile loss, which needs regressor='linear'")
        lo_a, hi_a = intervals
        outs = []
        for a in (lo_a, hi_a):
            qm = new_model(loss="quantile", quantile=a)
            _train(qm, y_fit, Xh, epochs)
            outs.append(np.clip(np.array(qm.forecast(horizon, xs=Xf), dtype=float) + base_f, 0, 100))
        lower = pd.Series(np.minimum(outs[0], point), index=idx, name="lower")
        upper = pd.Series(np.maximum(outs[1], point), index=idx, name="upper")
    elapsed = time.perf_counter() - t0

    cfg = dict(resolution=res, horizon=horizon, p=p, d=d, q=q, sp=sp, sd=sd, sq=sq, regressor=regressor, lr=lr,
               l2=l2, intercept_lr=intercept_lr, C=C, epochs=epochs, exog=exog, exog_lags=tuple(exog_lags),
               intervals=intervals, target=target)
    return ForecastResult(pd.Series(point, index=idx, name="forecast"), lower, upper, ds.index[-1], cfg,
                          elapsed, len(ds), res)

### Baselines and backtesting

A forecast is only useful if it beats something simple. Two baselines are computed for every forecast:

- **Seasonal naive**: the value one season (12 months / 52 weeks) earlier.
- **Climatology**: the average of the same month/week over the history window.

The model is tested with a **rolling-origin backtest**: for each origin the model is trained from scratch on the data up to that origin (optionally only the last `history_years`), forecasts the next `horizon` periods, and is compared with what really happened. Errors are in **percentage points of filling**. Origins are spaced 2 months (monthly) / 8 weeks (weekly) apart, so all seasons of the year are tested.

In [ ]:
def baseline_forecasts(hist, horizon, res):
    m = RESOLUTIONS[res]["m"]
    idx = future_index(hist.index[-1], horizon, res)
    ext = list(hist["y"].to_numpy(float))
    for _ in range(horizon):
        ext.append(ext[-m])                                     # seasonal naive (also works if horizon > m)
    clim = hist["y"].groupby(period_key(hist.index, res)).mean()
    clim_fc = pd.Series(period_key(idx, res)).map(clim).to_numpy(dtype=float)
    clim_fc = np.where(np.isnan(clim_fc), hist["y"].mean(), clim_fc)
    return np.array(ext[-horizon:]), clim_fc


def eval_origins(n_rows, horizon, n_origins, step, offset=0):
    # Positions (row numbers) of the forecast origins, newest = last possible one minus `offset`
    last = n_rows - 1 - horizon - offset
    return sorted(pos for pos in (last - k * step for k in range(n_origins)) if pos > 0)


def backtest(ds, horizon, n_origins=12, step=None, offset=0, history_years=None, exog=(), exog_lags=(),
             oracle=False, min_hist=None, **model_kw):
    # Rolling-origin backtest. Returns (detail DataFrame, summary dict).
    res = ds.attrs["resolution"]
    cfg = RESOLUTIONS[res]
    step = step or cfg["step"]
    min_hist = min_hist or (36 if res == "M" else 156)
    window = int(history_years * cfg["per_year"]) if history_years else None
    rows, fit_times = [], []

    for pos in eval_origins(len(ds), horizon, n_origins, step, offset):
        hist = ds.iloc[:pos + 1]
        if window:
            hist = hist.iloc[-window:]
        if len(hist) < min_hist:
            continue
        hist.attrs["resolution"] = res
        future = ds.iloc[pos + 1:pos + 1 + horizon]
        r = fit_and_forecast(hist, horizon, exog=exog, exog_lags=exog_lags, resolution=res,
                             exog_future="oracle" if oracle else "auto",
                             actual_future=future if oracle else None, **model_kw)
        sn, cl = baseline_forecasts(hist, horizon, res)
        fit_times.append(r.fit_seconds)
        for h in range(horizon):
            rows.append({"origin": hist.index[-1], "h": h + 1, "date": future.index[h], "y_true": future["y"].iloc[h],
                         "y_pred": r.forecast.iloc[h], "seasonal_naive": sn[h], "climatology": cl[h],
                         "lower": np.nan if r.lower is None else r.lower.iloc[h],
                         "upper": np.nan if r.upper is None else r.upper.iloc[h]})
    if not rows:
        raise ValueError("Not enough data for this combination of history length, horizon and evaluation window")

    det = pd.DataFrame(rows)
    err = lambda col: (det[col] - det["y_true"])
    mae, mae_sn, mae_cl = (err(c).abs().mean() for c in ("y_pred", "seasonal_naive", "climatology"))
    summary = {"mae": mae, "rmse": float(np.sqrt((err("y_pred") ** 2).mean())), "mae_seasonal_naive": mae_sn,
               "mae_climatology": mae_cl, "skill_vs_clim": 1 - mae / mae_cl, "skill_vs_naive": 1 - mae / mae_sn,
               "mean_fit_s": float(np.mean(fit_times)), "n_origins": int(det["origin"].nunique()), "n_train_last": len(hist)}
    if det["lower"].notna().any():
        inside = (det["y_true"] >= det["lower"]) & (det["y_true"] <= det["upper"])
        summary["coverage"] = float(inside.mean())
        summary["mean_width"] = float((det["upper"] - det["lower"]).mean())
    return det, summary

### The user-facing function

`predict_reservoir_fill()` is the function the Streamlit app will call. Inputs: region (`area_type`, `area_nr`), horizon in **months**, resolution, length of history, **data sources / exogenous variables**, all SNARIMAX and regressor parameters, and optional intervals. `DEFAULT_CONFIG` holds the default setting; it is updated with the best configuration in Cell 7.

In [ ]:
# Prior default (a sensible starting point); Cell 7 replaces it with the configuration that wins the experiments
DEFAULT_CONFIG = dict(resolution="M", horizon_months=6, history_years=15, target="anomaly",
                      p=2, d=0, q=0, sp=0, sd=0, sq=0,
                      regressor="linear", lr=0.01, l2=0.0, intercept_lr=0.1, C=1.0, epochs=1,
                      exog=(), exog_lags=(), intervals=None)

_RAW_CACHE = {}


def periods_for(horizon_months, res):
    return int(horizon_months) if res == "M" else int(round(horizon_months * 52 / 12))


def get_raw_sources(sources, area_type="EL", area_nr=3, weather_start="2005-01-01"):
    # Load (and memoise) the daily frames needed for the selected sources
    out = {}
    if "weather" in sources:
        key = ("weather", area_type, area_nr, weather_start)
        if key not in _RAW_CACHE:
            _RAW_CACHE[key] = fetch_weather_daily(area_type, area_nr, start=weather_start)
        out["weather_daily"] = _RAW_CACHE[key]
    if "currency" in sources:
        if "currency" not in _RAW_CACHE:
            _RAW_CACHE["currency"] = fetch_nb_exchange_rates(("EUR", "USD"), start="2000-01-01")
        out["fx_daily"] = _RAW_CACHE["currency"]
    if "entsoe" in sources:
        if area_type != "EL":
            raise ValueError("ENTSO-E net import is only available for electricity areas (EL)")
        key = ("entsoe", area_nr)
        if key not in _RAW_CACHE:
            _RAW_CACHE[key] = load_entsoe_net_import_daily(f"NO{int(area_nr)}")
        out["entsoe_daily"] = _RAW_CACHE[key]
    return out


def predict_reservoir_fill(area_type="EL", area_nr=3, horizon_months=None, resolution=None, history_years=None,
                           sources=None, exog=None, exog_lags=None, target=None, p=None, d=None, q=None, sp=None, sd=None, sq=None,
                           regressor=None, lr=None, l2=None, intercept_lr=None, C=None, epochs=None, intervals=None,
                           nve=None, raw=None, verbose=True):
    # Forecast the filling (%) of ONE region for the coming months with River SNARIMAX.
    #   area_type, area_nr : region in Magasinstatistikk, e.g. ("EL", 3)
    #   horizon_months     : how many months ahead (weekly resolution converts to weeks)
    #   resolution         : "M" monthly or "W" weekly
    #   history_years      : length of the training history (None = everything available)
    #   sources / exog     : data sources ("weather", "currency", "entsoe") and/or the exact variables
    #                        from EXOG_CATALOG. exog alone picks the variables (and thus the sources);
    #                        sources alone uses all variables of those sources.
    #   target             : "anomaly" (model deviations from the climatology) or "level"
    #   p,d,q,sp,sd,sq     : SNARIMAX orders; regressor, lr, l2, intercept_lr, C, epochs: online learner
    #   intervals          : e.g. (0.1, 0.9) for an 80 % interval from quantile regression
    #   nve / raw          : optionally pre-loaded NVE data / daily source frames (avoids reloading)
    # Anything left as None takes its value from DEFAULT_CONFIG. Returns (ForecastResult, dataset).
    given = dict(horizon_months=horizon_months, resolution=resolution, history_years=history_years, p=p, d=d, q=q,
                 sp=sp, sd=sd, sq=sq, regressor=regressor, lr=lr, l2=l2, intercept_lr=intercept_lr, C=C,
                 epochs=epochs, exog_lags=exog_lags, intervals=intervals, target=target)
    cfg = {**DEFAULT_CONFIG, **{k: v for k, v in given.items() if v is not None}}
    if history_years == 0:
        cfg["history_years"] = None

    chosen = tuple(exog) if exog else tuple(cfg["exog"]) if sources is None else ()
    if sources is not None:
        allowed = [c for c, (s, _) in EXOG_CATALOG.items() if s in tuple(sources)]
        chosen = tuple(exog) if exog else tuple(allowed)
        bad = [c for c in chosen if c not in allowed]
        if bad:
            raise ValueError(f"{bad} do not belong to the selected sources {tuple(sources)}")
    needed_sources = sorted({EXOG_CATALOG[c][0] for c in chosen if c in EXOG_CATALOG})

    nve = nve if nve is not None else load_nve_raw()
    fill_weekly = get_fill_series(nve, area_type, area_nr)
    raw = raw if raw is not None else get_raw_sources(needed_sources, area_type, area_nr)
    ds = build_dataset(fill_weekly, cfg["resolution"], chosen, **raw)

    if cfg["history_years"]:
        ds = ds.iloc[-int(cfg["history_years"] * RESOLUTIONS[cfg["resolution"]]["per_year"]):]
        ds.attrs["resolution"] = cfg["resolution"]
    h = periods_for(cfg["horizon_months"], cfg["resolution"])
    kw = {k: cfg[k] for k in ("p", "d", "q", "sp", "sd", "sq", "regressor", "lr", "l2", "intercept_lr", "C", "epochs")}
    res = fit_and_forecast(ds, h, exog=chosen, exog_lags=cfg["exog_lags"], intervals=cfg["intervals"],
                           resolution=cfg["resolution"], target=cfg["target"], **kw)
    if verbose:
        print(f"{area_type} {area_nr}: trained on {res.n_train} {RESOLUTIONS[res.resolution]['label']} points up to "
              f"{res.origin.date()} in {res.fit_seconds:.2f}s; forecast {h} step(s) ahead; exog = {list(chosen) or 'none'}")
    return res, ds


def plot_forecast(res, ds, years_back=6, title=None):
    fig, ax = plt.subplots(figsize=(12, 4.5))
    n_back = years_back * RESOLUTIONS[res.resolution]["per_year"]
    ax.plot(ds.index[-n_back:], ds["y"].iloc[-n_back:], color="steelblue", label="observed")
    ax.plot(res.forecast.index, res.forecast, color="crimson", marker="o", ms=4, label="forecast")
    if res.lower is not None:
        ax.fill_between(res.forecast.index, res.lower, res.upper, color="crimson", alpha=0.2, label="interval")
    ax.axvline(res.origin, color="grey", linestyle="--", linewidth=0.8)
    ax.set_ylabel("Filling (%)"); ax.set_ylim(0, 100); ax.grid(True, alpha=0.3); ax.legend()
    ax.set_title(title or f"Reservoir filling forecast (origin {res.origin.date()})")
    plt.tight_layout(); plt.show()


# Smoke test with the prior default and no exogenous data
res0, ds0 = predict_reservoir_fill("EL", 3)
display(res0.forecast.round(1).to_frame())
plot_forecast(res0, ds0)

## Cell 7 – Testing different settings and choosing the default

The experiments are run **stage by stage** (a greedy search), because a full grid over all parameters would be far too large. Every stage keeps the winner of the previous stage fixed and varies one group of settings:

0. target transformation (anomaly vs level),
1. time resolution (monthly vs weekly), judged on error *and* run time,
2. model orders `p, sp`, then `d, sd`, then `q`,
3. the online regressor and its learning settings,
4. length of the training history,
5. exogenous variables (weather, currency, lags), plus the oracle diagnostic.

To avoid choosing a setting that is only good by luck, two **disjoint backtest windows** are used: the stages are tuned on 24 origins (monthly: 2 months apart, about 4 years) *before* the most recent 24 origins, and the chosen default is finally judged on the **most recent 24 origins** (test window), which no stage has used for decisions.

**Selection rule (parsimony).** Within each stage the variants are listed from *simplest* to *most complex* (monthly before weekly, fewer lags/variables/orders first, default learner before tweaks). The winner is the **simplest variant whose MAE is within 5 % of the best MAE**, and whose fit time stays under 3 seconds (the app needs fast answers). A bigger or slower model therefore has to be *clearly* better to be chosen, which also protects against picking a variable that only wins by noise.

In [ ]:
EXP_N_ORIGINS   = 24        # backtest origins per window (monthly: 2 months apart = about 4 years)
FIT_TIME_BUDGET = 3.0       # seconds per forecast that is acceptable in the Streamlit app
TOLERANCE       = 0.05      # a variant within 5 % of the best MAE counts as equally good (then the simplest wins)
MODEL_KEYS = ["p", "d", "q", "sp", "sd", "sq", "regressor", "lr", "l2", "intercept_lr", "C", "epochs", "target"]

RAW_ALL = dict(weather_daily=weather_daily, fx_daily=fx, entsoe_daily=None)
_ds_cache = {}


def dataset_for(cfg):
    key = (cfg["resolution"], tuple(cfg["exog"]))
    if key not in _ds_cache:
        _ds_cache[key] = build_dataset(fill_weekly, cfg["resolution"], cfg["exog"], **RAW_ALL)
    return _ds_cache[key]


def evaluate_cfg(cfg, window="tune", oracle=False, intervals=None):
    res = cfg["resolution"]
    ds = dataset_for(cfg)
    offset = 0 if window == "test" else EXP_N_ORIGINS * RESOLUTIONS[res]["step"]
    return backtest(ds, periods_for(cfg["horizon_months"], res), n_origins=EXP_N_ORIGINS, offset=offset,
                    history_years=cfg["history_years"], exog=cfg["exog"], exog_lags=cfg["exog_lags"], oracle=oracle,
                    intervals=intervals, **{k: cfg[k] for k in MODEL_KEYS})


def run_stage(title, base_cfg, variants, diagnostics=()):
    # Evaluate each variant (a dict of changes to base_cfg) on the tuning window and return a results table
    rows = []
    print(f"\n=== {title}")
    for name, delta in variants.items():
        try:
            _, s = evaluate_cfg({**base_cfg, **delta}, "tune", oracle=name in diagnostics)
            rows.append({"variant": name, **s, "diagnostic": name in diagnostics})
        except Exception as e:
            print(f"  skipped '{name}': {str(e)[:110]}")
    table = pd.DataFrame(rows)
    shown = ["variant", "mae", "rmse", "mae_climatology", "skill_vs_clim", "mean_fit_s", "n_origins"]
    display(table.sort_values("mae")[shown + ["diagnostic"]].round(3).reset_index(drop=True))
    return table


def choose(table, variants):
    # Parsimony rule: variants are listed simplest-first. Pick the first one whose MAE is within TOLERANCE of the
    # best MAE (and within the time budget). Diagnostics never win.
    t = table[~table["diagnostic"]]
    ok = t[t["mean_fit_s"] <= FIT_TIME_BUDGET]
    ok = ok if len(ok) else t
    best = ok["mae"].min()
    pick = ok[ok["mae"] <= best * (1 + TOLERANCE)].iloc[0]["variant"]          # table keeps the order of `variants`
    print(f"-> chosen: {pick}  (MAE {ok.set_index('variant').loc[pick, 'mae']:.3f}; best MAE in this stage {best:.3f})")
    return pick, variants[pick]


cfg = {**DEFAULT_CONFIG}
stage_log = []          # (stage, chosen variant, mae, mae of climatology baseline, fit seconds)


def record(stage, table, pick):
    r = table.set_index("variant").loc[pick]
    stage_log.append({"stage": stage, "chosen": pick, "mae": r["mae"], "mae_climatology": r["mae_climatology"],
                      "mean_fit_s": r["mean_fit_s"]})

In [ ]:
# Stage 0 - target transformation: model the raw level, or the anomaly relative to the climatology?
v = {"level (p=2, sp=1)":   {"target": "level",   "sp": 1},
     "level (p=2, sd=1)":   {"target": "level",   "sp": 0, "sd": 1},
     "anomaly (p=2)":       {"target": "anomaly", "sp": 0}}
t = run_stage("Stage 0: target transformation", cfg, v)
pick, delta = choose(t, v); record("target", t, pick); cfg.update(delta)

# Stage 1 - time resolution: monthly vs weekly (weekly has 52-step seasonality and ~4x more data points).
# Monthly is listed first (simpler and faster): weekly has to be clearly better to be chosen.
v = {"M (p=2)": {"resolution": "M"},
     "W (p=2)": {"resolution": "W"},
     "W (p=4)": {"resolution": "W", "p": 4}}
t = run_stage("Stage 1: time resolution", cfg, v)
pick, delta = choose(t, v); record("resolution", t, pick); cfg.update(delta)

In [ ]:
# Stage 2 - model orders. Non-seasonal and seasonal AR order first ...
ps = (1, 2, 3, 6) if cfg["resolution"] == "M" else (2, 4, 8, 13)
combos = sorted(((p_, sp_) for p_ in ps for sp_ in (0, 1, 2)), key=lambda x: (x[0] + x[1], x[0]))   # simplest first
v = {f"p={p_}, sp={sp_}": {"p": p_, "sp": sp_} for p_, sp_ in combos}
t = run_stage("Stage 2a: AR orders (p, sp)", cfg, v)
pick, delta = choose(t, v); record("p, sp", t, pick); cfg.update(delta)

# ... then differencing ...
v = {f"d={d_}, sd={sd_}": {"d": d_, "sd": sd_} for d_ in (0, 1) for sd_ in (0, 1)}
t = run_stage("Stage 2b: differencing (d, sd)", cfg, v)
pick, delta = choose(t, v); record("d, sd", t, pick); cfg.update(delta)

# ... then the moving-average order
v = {f"q={q_}": {"q": q_} for q_ in (0, 1, 2)}
t = run_stage("Stage 2c: MA order (q)", cfg, v)
pick, delta = choose(t, v); record("q", t, pick); cfg.update(delta)

In [ ]:
# Stage 3 - the online regressor and its learning settings (cfg keeps the orders chosen above)
v = {"linear (lr 0.01)": {},
     "linear lr 0.003": {"lr": 0.003}, "linear lr 0.03": {"lr": 0.03},
     "linear l2 0.01": {"l2": 0.01}, "linear l2 0.1": {"l2": 0.1},
     "linear intercept_lr 0.03": {"intercept_lr": 0.03}, "linear intercept_lr 0.3": {"intercept_lr": 0.3},
     "linear 3 epochs": {"epochs": 3}, "linear 5 epochs": {"epochs": 5},
     "passive-aggressive C=0.1": {"regressor": "pa", "C": 0.1},
     "passive-aggressive C=1": {"regressor": "pa", "C": 1.0},
     "passive-aggressive C=10": {"regressor": "pa", "C": 10.0},
     "hoeffding tree": {"regressor": "tree"}}
t = run_stage("Stage 3: regressor and learning settings", cfg, v)
pick, delta = choose(t, v); record("regressor", t, pick); cfg.update(delta)

# Stage 4 - length of the training history
v = {f"{y_} years": {"history_years": y_} for y_ in (5, 8, 12, 16, 20)}
v["all data"] = {"history_years": None}
t = run_stage("Stage 4: history length", cfg, v)
pick, delta = choose(t, v); record("history", t, pick); cfg.update(delta)

In [ ]:
# Stage 5 - exogenous variables. Weather is unknown in the future, so it enters as a climatology (see Cell 6).
W4 = ("temperature_2m_mean", "precipitation_sum", "snowfall_sum", "wind_speed_10m_max")
v = {"none": {"exog": ()},
     "temperature": {"exog": ("temperature_2m_mean",)},
     "temperature + precipitation": {"exog": ("temperature_2m_mean", "precipitation_sum")},
     "temp + precip + snowfall": {"exog": W4[:3]},
     "all four weather variables": {"exog": W4},
     "EUR/NOK": {"exog": ("eur_nok",)},
     "EUR/NOK + USD/NOK": {"exog": ("eur_nok", "usd_nok")},
     "weather(3) + EUR/NOK": {"exog": W4[:3] + ("eur_nok",)}}
t5 = run_stage("Stage 5a: exogenous sources", cfg, v)
pick, delta = choose(t5, v); record("exogenous", t5, pick); cfg.update(delta)

# Lagged exogenous values (snow and rain reach the reservoirs with a delay) for the chosen set
if cfg["exog"]:
    lag_sets = {"no lags": (), "lag 1": (1,), "lags 1-2": (1, 2), "lags 1, 3, 6": (1, 3, 6)} if cfg["resolution"] == "M" \
          else {"no lags": (), "lag 4": (4,), "lags 4, 8": (4, 8), "lags 4, 13, 26": (4, 13, 26)}
    v = {name: {"exog_lags": lags} for name, lags in lag_sets.items()}
    t = run_stage("Stage 5b: lags of the exogenous variables", cfg, v)
    pick, delta = choose(t, v); record("exog lags", t, pick); cfg.update(delta)

# Diagnostic (never chosen): what if the FUTURE weather were known? Shows how much the unknown future costs.
if cfg["exog"]:
    v = {"chosen setting, assumed future exog": {}, "chosen setting, TRUE future exog (oracle)": {}}
    t = run_stage("Diagnostic: oracle exogenous future", cfg, v, diagnostics=("chosen setting, TRUE future exog (oracle)",))

# ENTSO-E flow: only 2024-2025 are stored, which is too short for a multi-year backtest
try:
    e = load_entsoe_net_import_daily("NO3")
    n_months = len(aggregate_daily(e, "M"))
    print(f"\nENTSO-E net import: {n_months} complete months available in MongoDB "
          f"(the two backtest windows need about {36 + 6 + 2 * EXP_N_ORIGINS * RESOLUTIONS['M']['step']} months) -> not usable as a predictor yet")
except Exception as err:
    print("\nENTSO-E check skipped:", str(err)[:120])

### Held-out evaluation of the chosen default

The chosen configuration (and, for comparison, the prior default and the two baselines) is evaluated on the **test window**, the 12 most recent origins that no stage used for decisions. The error is shown per forecast step, for horizons of 3, 6 and 12 months.

In [ ]:
final_cfg = {**cfg}
prior_cfg = {**DEFAULT_CONFIG}
rows = []
curves = {}
level_cfg = {**DEFAULT_CONFIG, "target": "level", "sp": 1}        # plain SNARIMAX on the raw level, for reference
for label, c in [("level model (p=2, sp=1)", level_cfg), ("prior default", prior_cfg), ("chosen default", final_cfg)]:
    for hm in (3, 6, 12):
        det, s = evaluate_cfg({**c, "horizon_months": hm}, "test")
        rows.append({"config": label, "horizon (months)": hm, "MAE": s["mae"], "MAE seasonal naive": s["mae_seasonal_naive"],
                     "MAE climatology": s["mae_climatology"], "skill vs climatology": s["skill_vs_clim"],
                     "fit time (s)": s["mean_fit_s"]})
        if hm == 12:
            curves[label] = det
test_table = pd.DataFrame(rows)
display(test_table.round(3))

# Error per step ahead for the 12-month horizon
fig, ax = plt.subplots(figsize=(9, 4))
for label, det in curves.items():
    ax.plot(det.groupby("h").apply(lambda g: (g["y_pred"] - g["y_true"]).abs().mean()), marker="o", label=label)
d = curves["chosen default"]
ax.plot(d.groupby("h").apply(lambda g: (g["climatology"] - g["y_true"]).abs().mean()), "--", color="grey", label="climatology")
ax.plot(d.groupby("h").apply(lambda g: (g["seasonal_naive"] - g["y_true"]).abs().mean()), ":", color="black", label="seasonal naive")
ax.set_xlabel("Forecast step ahead (resolution periods)"); ax.set_ylabel("MAE (percentage points)")
ax.set_title("Held-out test window: error vs. forecast step"); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Does the chosen default beat the simple baselines on data it was not tuned on?
chosen_rows = test_table[test_table["config"] == "chosen default"]
for _, r in chosen_rows.iterrows():
    verdict = "beats" if r["MAE"] < r["MAE climatology"] else "does NOT beat"
    print(f"Horizon {int(r['horizon (months)']):2d} months: chosen default {verdict} the climatology baseline "
          f"(MAE {r['MAE']:.2f} vs {r['MAE climatology']:.2f}; seasonal naive {r['MAE seasonal naive']:.2f})")

# Summary of the greedy search
display(pd.DataFrame(stage_log).round(3))

In [ ]:
# Confidence intervals (bonus): quantile regression with optim.losses.Quantile, 80 % nominal interval
det_i, s_i = evaluate_cfg({**final_cfg, "regressor": "linear"}, "test", intervals=(0.1, 0.9))
print(f"Intervals (0.1, 0.9) on the test window: coverage {s_i['coverage']:.0%} (nominal 80 %), "
      f"mean width {s_i['mean_width']:.1f} percentage points, point-forecast MAE {s_i['mae']:.2f}")

last_origin = det_i["origin"].max()
ex = det_i[det_i["origin"] == last_origin]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ex["date"], ex["y_true"], color="steelblue", marker="o", label="actual")
ax.plot(ex["date"], ex["y_pred"], color="crimson", marker="o", label="forecast")
ax.fill_between(ex["date"], ex["lower"], ex["upper"], color="crimson", alpha=0.2, label="10-90 % interval")
ax.set_ylabel("Filling (%)"); ax.set_title(f"Last backtest origin ({last_origin.date()})"); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

### Default setting for the Streamlit app

The winning configuration becomes `DEFAULT_CONFIG` (so `predict_reservoir_fill()` uses it when nothing else is specified) and is saved to `forecast_defaults.json`, which the Streamlit page can read to preset its controls.

In [ ]:
DEFAULT_CONFIG = {**final_cfg}
DEFAULT_CONFIG["intervals"] = None

with open("forecast_defaults.json", "w") as f:
    json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in DEFAULT_CONFIG.items()}, f, indent=2)
print("Default configuration for the Streamlit app:")
print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in DEFAULT_CONFIG.items()}, indent=2))

# The function now works with the new defaults, and any setting can still be overridden:
res_final, ds_final = predict_reservoir_fill("EL", 3, intervals=(0.1, 0.9))
display(pd.concat([res_final.lower, res_final.forecast, res_final.upper], axis=1).round(1))
plot_forecast(res_final, ds_final, title=f"EL 3 - forecast with the chosen default (origin {res_final.origin.date()})")

# Example of overriding settings: weekly resolution, 3 months ahead, 8 years of history
res_w, ds_w = predict_reservoir_fill("EL", 3, resolution="W", horizon_months=3, history_years=8)
print(res_w.forecast.round(1).head(3).to_dict())

## Findings and log

**Experiment findings** – to be written after the run, from the tables above: which resolution won and why (error vs run time), which orders and regressor settings mattered, how much the history length and the exogenous variables changed the error, how the chosen default compares with the climatology and seasonal-naive baselines on the held-out window, and the interval coverage.

**Log (300–500 words)** – to be written (must also mention the bonus task chosen for the Streamlit app).

**AI usage** – to be written (brief description).